[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module5/08-ensemble-methods.ipynb)

# Module 5 — Lesson 8: Ensemble Methods

**Module:** 5 — Machine Learning Foundations | **Time:** 30 minutes

## Learning Objectives

By the end of this lesson you will be able to:

- Explain and contrast bagging and boosting strategies
- Train XGBoost and LightGBM classifiers with early stopping
- Build stacking ensembles with `StackingClassifier` and a meta-learner
- Create voting ensembles (hard and soft voting)
- Implement manual blending using a holdout validation set
- Compare all methods in a comprehensive benchmark table

In [ ]:
!pip install -q xgboost lightgbm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier,
                               AdaBoostClassifier, GradientBoostingClassifier,
                               StackingClassifier, VotingClassifier)
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print('All libraries loaded. XGBoost and LightGBM ready.')

## 1. Bagging vs Boosting — Concepts

| Strategy | Idea | Key models | Variance | Bias |
|---|---|---|---|---|
| **Bagging** | Train many models on bootstrap samples; aggregate by voting/averaging | Random Forest, BaggingClassifier | Reduces | Unchanged |
| **Boosting** | Train models sequentially; each focuses on previous errors | AdaBoost, GradientBoosting, XGBoost | Slight increase | Reduces |

Bagging reduces **variance** (overfitting). Boosting reduces **bias** (underfitting) but can overfit if too many iterations are used.

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# A single decision tree (baseline)
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)

# Bagging: BaggingClassifier wraps any base estimator
bag = BaggingClassifier(estimator=DecisionTreeClassifier(random_state=42),
                         n_estimators=100, bootstrap=True, random_state=42, n_jobs=-1)
bag.fit(X_train, y_train)

print('Bagging vs Single Tree:')
print(f'  Single Decision Tree  — Train: {accuracy_score(y_train, dt.predict(X_train)):.4f}  Test: {accuracy_score(y_test, dt.predict(X_test)):.4f}')
print(f'  BaggingClassifier(100)— Train: {accuracy_score(y_train, bag.predict(X_train)):.4f}  Test: {accuracy_score(y_test, bag.predict(X_test)):.4f}')
print('\nKey insight: bagging reduces the test error by averaging out individual tree variance.')

## 2. Random Forest — Bagging with Feature Randomness

In [ ]:
# Random Forest scalability: n_estimators effect
n_estimators_range = [1, 5, 10, 25, 50, 100, 200, 300]
train_accs, test_accs = [], []

for n in n_estimators_range:
    rf = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, rf.predict(X_train)))
    test_accs.append(accuracy_score(y_test, rf.predict(X_test)))

plt.figure(figsize=(9, 4))
plt.plot(n_estimators_range, train_accs, 'o-', label='Train', color='steelblue')
plt.plot(n_estimators_range, test_accs,  's-', label='Test',  color='tomato')
plt.xlabel('Number of Trees')
plt.ylabel('Accuracy')
plt.title('Random Forest — Trees vs Accuracy')
plt.legend()
plt.tight_layout()
plt.show()
print(f'\nRandom Forest (300 trees) — Test: {test_accs[-1]:.4f}')

## 3. Boosting — AdaBoost and Gradient Boosting

In [ ]:
# AdaBoost
ada = AdaBoostClassifier(n_estimators=200, learning_rate=0.5,
                          estimator=DecisionTreeClassifier(max_depth=1),
                          random_state=42)
ada.fit(X_train, y_train)

# GradientBoosting
gb = GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                  learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)

# Track staged accuracy for GradientBoosting
gb_staged_test = [accuracy_score(y_test, y_pred) for y_pred in gb.staged_predict(X_test)]
gb_staged_train = [accuracy_score(y_train, y_pred) for y_pred in gb.staged_predict(X_train)]

plt.figure(figsize=(9, 4))
plt.plot(gb_staged_train, label='GradBoost Train', color='steelblue', lw=1.5)
plt.plot(gb_staged_test,  label='GradBoost Test',  color='tomato',    lw=1.5)
plt.xlabel('Number of Trees')
plt.ylabel('Accuracy')
plt.title('Gradient Boosting — Learning Curve per Tree')
plt.legend()
plt.tight_layout()
plt.show()

print(f'AdaBoost     Test Acc: {accuracy_score(y_test, ada.predict(X_test)):.4f}')
print(f'GradBoosting Test Acc: {accuracy_score(y_test, gb.predict(X_test)):.4f}')

## 4. XGBoost with Early Stopping

XGBoost (Extreme Gradient Boosting) is an optimised gradient boosting library with:
- Regularisation (L1/L2) built in
- Sparse data handling
- Early stopping to prevent overfitting

In [ ]:
# Further split training data for early stopping eval_set
X_tr_xgb, X_val_xgb, y_tr_xgb, y_val_xgb = train_test_split(
    X_train, y_train, test_size=0.15, stratify=y_train, random_state=42
)

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    early_stopping_rounds=20,
    random_state=42,
    verbosity=0
)
xgb.fit(X_tr_xgb, y_tr_xgb,
        eval_set=[(X_val_xgb, y_val_xgb)],
        verbose=False)

print(f'XGBoost — best iteration: {xgb.best_iteration}')
print(f'XGBoost — Test Accuracy:  {accuracy_score(y_test, xgb.predict(X_test)):.4f}')
print(f'XGBoost — Test AUC-ROC:   {roc_auc_score(y_test, xgb.predict_proba(X_test)[:,1]):.4f}')

# Feature importances
fig, ax = plt.subplots(figsize=(10, 5))
imp = xgb.feature_importances_
idx = np.argsort(imp)[::-1][:15]
ax.bar(range(15), imp[idx], color='steelblue', alpha=0.8)
ax.set_xticks(range(15))
ax.set_xticklabels([data.feature_names[i] for i in idx], rotation=40, ha='right', fontsize=8)
ax.set_ylabel('Importance (F-score)')
ax.set_title('XGBoost Feature Importances (Top 15)')
plt.tight_layout()
plt.show()

## 5. LightGBM

LightGBM (Light Gradient Boosting Machine) uses histogram-based splits and leaf-wise (rather than level-wise) tree growth, making it significantly faster than XGBoost on large datasets.

In [ ]:
lgbm = LGBMClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)
lgbm.fit(X_tr_xgb, y_tr_xgb,
          eval_set=[(X_val_xgb, y_val_xgb)],
          callbacks=[]
          )

print(f'LightGBM — Test Accuracy: {accuracy_score(y_test, lgbm.predict(X_test)):.4f}')
print(f'LightGBM — Test AUC-ROC:  {roc_auc_score(y_test, lgbm.predict_proba(X_test)[:,1]):.4f}')

# Cross-validation comparison
for name, model in [('XGBoost', XGBClassifier(n_estimators=100, random_state=42, verbosity=0, eval_metric='logloss')),
                    ('LightGBM', LGBMClassifier(n_estimators=100, random_state=42, verbose=-1))]:
    cv = cross_val_score(model, X, y, cv=StratifiedKFold(5), scoring='roc_auc')
    print(f'{name:10s} CV AUC: {cv.mean():.4f} ± {cv.std():.4f}')

## 6. Voting Classifier — Hard and Soft Voting

- **Hard voting** — each model casts a vote; majority wins
- **Soft voting** — models output probabilities; the class with the highest mean probability wins (usually better)

In [ ]:
lr_v  = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])
rf_v  = RandomForestClassifier(n_estimators=100, random_state=42)
xgb_v = XGBClassifier(n_estimators=100, random_state=42, verbosity=0, eval_metric='logloss')

voting_hard = VotingClassifier(
    estimators=[('lr', lr_v), ('rf', rf_v), ('xgb', xgb_v)],
    voting='hard'
)
voting_soft = VotingClassifier(
    estimators=[('lr', lr_v), ('rf', rf_v), ('xgb', xgb_v)],
    voting='soft'
)

for name, model in [('Hard Voting', voting_hard), ('Soft Voting', voting_soft)]:
    model.fit(X_train, y_train)
    print(f'{name}: Test Acc={accuracy_score(y_test, model.predict(X_test)):.4f}')

## 7. Stacking with StackingClassifier

Stacking uses a **meta-learner** trained on the out-of-fold predictions of base learners. It is more flexible than voting because the meta-learner can learn the optimal combination.

In [ ]:
base_learners = [
    ('lr',  Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])),
    ('rf',  RandomForestClassifier(n_estimators=100, random_state=42)),
    ('xgb', XGBClassifier(n_estimators=100, random_state=42, verbosity=0, eval_metric='logloss')),
    ('svm', Pipeline([('sc', StandardScaler()), ('clf', SVC(kernel='rbf', probability=True))])),
]

meta_learner = LogisticRegression(max_iter=1000)

stacking = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5,
    passthrough=False,
    n_jobs=-1
)
stacking.fit(X_train, y_train)

print(f'Stacking — Test Accuracy: {accuracy_score(y_test, stacking.predict(X_test)):.4f}')
print(f'Stacking — Test AUC-ROC:  {roc_auc_score(y_test, stacking.predict_proba(X_test)[:,1]):.4f}')

## 8. Blending — Holdout-Based Stacking

Blending is a simpler alternative to stacking where base learners are trained on one partition and a meta-learner is trained on their predictions on a held-out blend set.

In [ ]:
# Split: 60% train, 20% blend, 20% test
X_tr_b, X_blend_test, y_tr_b, y_blend_test = train_test_split(X_train, y_train,
                                                               test_size=0.25, stratify=y_train, random_state=1)
X_blend, X_te_blend, y_blend, y_te_blend = train_test_split(X_blend_test, y_blend_test,
                                                             test_size=0.5, stratify=y_blend_test, random_state=2)

base_models = {
    'lr':  Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))]),
    'rf':  RandomForestClassifier(n_estimators=100, random_state=42),
    'xgb': XGBClassifier(n_estimators=100, random_state=42, verbosity=0, eval_metric='logloss')
}

# Train base models on training set; predict on blend set
blend_train = np.zeros((len(y_blend), len(base_models)))
blend_test  = np.zeros((len(y_te_blend), len(base_models)))

for i, (name, model) in enumerate(base_models.items()):
    model.fit(X_tr_b, y_tr_b)
    blend_train[:, i] = model.predict_proba(X_blend)[:, 1]
    blend_test[:, i]  = model.predict_proba(X_te_blend)[:, 1]

# Train meta-learner on blend predictions
meta = LogisticRegression()
meta.fit(blend_train, y_blend)
blend_final_preds = meta.predict(blend_test)
blend_final_probs = meta.predict_proba(blend_test)[:, 1]

print(f'Blending — Test Accuracy: {accuracy_score(y_te_blend, blend_final_preds):.4f}')
print(f'Blending — Test AUC-ROC:  {roc_auc_score(y_te_blend, blend_final_probs):.4f}')

## 9. Full Benchmark Comparison

In [ ]:
benchmark_models = [
    ('Single Decision Tree',  DecisionTreeClassifier(random_state=42)),
    ('BaggingClassifier',     BaggingClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('Random Forest',         RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ('AdaBoost',              AdaBoostClassifier(n_estimators=200, random_state=42)),
    ('Gradient Boosting',     GradientBoostingClassifier(n_estimators=200, random_state=42)),
    ('XGBoost',               XGBClassifier(n_estimators=200, random_state=42, verbosity=0, eval_metric='logloss')),
    ('LightGBM',              LGBMClassifier(n_estimators=200, random_state=42, verbose=-1)),
    ('Hard Voting',           voting_hard),
    ('Soft Voting',           voting_soft),
    ('Stacking',              stacking),
]

results = []
for name, model in benchmark_models:
    try:
        cv_sc = cross_val_score(model, X, y, cv=StratifiedKFold(5), scoring='accuracy', n_jobs=-1)
        model.fit(X_train, y_train)
        test_acc = accuracy_score(y_test, model.predict(X_test))
        try:
            proba = model.predict_proba(X_test)[:, 1]
            auc   = roc_auc_score(y_test, proba)
        except Exception:
            auc = float('nan')
        results.append({'Model': name,
                        'CV Mean': round(cv_sc.mean(), 4),
                        'CV Std':  round(cv_sc.std(), 4),
                        'Test Acc': round(test_acc, 4),
                        'AUC-ROC': round(auc, 4)})
    except Exception as e:
        results.append({'Model': name, 'CV Mean': 'ERR', 'CV Std': 'ERR',
                        'Test Acc': 'ERR', 'AUC-ROC': 'ERR'})

bench_df = pd.DataFrame(results).sort_values('Test Acc', ascending=False).reset_index(drop=True)
print('=== Ensemble Methods Benchmark ===')
print(bench_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
numeric_df = bench_df[pd.to_numeric(bench_df['Test Acc'], errors='coerce').notna()].copy()
numeric_df['Test Acc'] = pd.to_numeric(numeric_df['Test Acc'])
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(numeric_df)))
ax.barh(numeric_df['Model'], numeric_df['Test Acc'], color=colors[::-1], alpha=0.85)
ax.set_xlabel('Test Accuracy')
ax.set_title('Ensemble Methods — Benchmark Comparison')
ax.set_xlim(0.9, 1.0)
plt.tight_layout()
plt.show()

## Practice Exercises

**Exercise 1 — XGBoost Hyperparameter Tuning**
Using `load_breast_cancer()`, run `RandomizedSearchCV` on `XGBClassifier` over the following distributions: `n_estimators` in [50, 100, 200], `max_depth` in [3, 4, 5, 6], `learning_rate` in uniform(0.01, 0.2), `subsample` in uniform(0.6, 0.4). Use 5-fold stratified CV with 20 iterations. Report the best parameters and test AUC-ROC.

**Exercise 2 — Stacking with Alternative Meta-Learners**
Build two `StackingClassifier` instances with the same base learners but different final estimators: (a) `LogisticRegression` and (b) `GradientBoostingClassifier`. Compare their 5-fold CV accuracy. Does a more powerful meta-learner always win? Explain why or why not.

**Exercise 3 — Diversity Analysis**
Train 5 diverse classifiers on the breast cancer dataset. Compute the pairwise disagreement rate between every pair (fraction of test samples where they predict differently). Create a 5x5 heatmap of disagreement rates. Which pairs are most diverse? Does diversity correlate with the performance of a soft-voting ensemble built from all 5?